In [ ]:
import wave
from piper import PiperVoice


voice = PiperVoice.load(r"C:\Users\User\OneDrive\Desktop\en_US-lessac-medium.onnx")

text = "Hello, I am Avanti."


with wave.open("output.wav", "wb") as wav_file:

    voice.synthesize_wav(
        text,
        wav_file
    )

print("Speech generated!")

Speech generated!


In [7]:
from faster_whisper import WhisperModel
import sounddevice as sd
import re
from LLM import get_completion
from tts import speak
import numpy as np

SAMPLE_RATE = 16000

sd.default.device = 3


# Initialize Whisper

model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)



# Record audio

def record_audio(duration):

    audio = sd.rec(
        int(duration * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32"
    )

    sd.wait()

    return audio.flatten()



# Transcribe

def transcribe_audio(audio):

    segments, info = model.transcribe(
        audio,
        language="en"
    )

    text = ""

    for segment in segments:
        text += segment.text

    text = re.sub(r"[^\w\s]", "", text)

    return text.lower().strip()

# -------------------------
# Listen until silence
# -------------------------

CHUNK_SIZE = 1024
SILENCE_THRESHOLD = 0.01
SILENCE_DURATION = 1.0


CHUNK_SIZE = 1024
SILENCE_THRESHOLD = 0.01
SILENCE_DURATION = 1.0
MAX_COMMAND_TIME = 10


def listen_until_silence():

    print("Listening...")

    audio_chunks = []

    speaking = False
    silence_time = 0
    total_time = 0

    while total_time < MAX_COMMAND_TIME:

        audio = sd.rec(
            CHUNK_SIZE,
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32"
        )

        sd.wait()

        audio = audio.flatten()

        rms = np.sqrt(np.mean(audio ** 2))

        chunk_duration = CHUNK_SIZE / SAMPLE_RATE
        total_time += chunk_duration

        if rms > SILENCE_THRESHOLD:

            speaking = True
            silence_time = 0

        elif speaking:

            silence_time += chunk_duration

        # Always save the audio after speech starts
        if speaking:
            audio_chunks.append(audio)

        # Stop after 1 second of silence
        if speaking and silence_time >= SILENCE_DURATION:

            print("Finished listening.")

            break

    if not audio_chunks:
        return np.array([], dtype=np.float32)

    return np.concatenate(audio_chunks)





active = False

while True:

    if not active:

        print("Waiting for Hey Avanti...")

        audio = record_audio(5)

        text = transcribe_audio(audio)

        print("Heard:", text)

        if "hey avanti" in text:

            print("HEY AVANTI DETECTED!")

            active = True

            speak("How can I help you?")


    else:

        command_audio = listen_until_silence()

        command = transcribe_audio(command_audio)

        print("Command:", command)
        if not command:
            continue

        if "bye avanti" in command:

            print("BYE AVANTI")

            speak("Goodbye.")

            active = False

        else:

            response = get_completion(command)

            speak(response)



Waiting for Hey Avanti...
Heard: you
Waiting for Hey Avanti...


KeyboardInterrupt: 

In [4]:
import soundfile as sf

command_audio = listen_until_silence()

sf.write(
    "debug_command.wav",
    command_audio,
    SAMPLE_RATE
)

command = transcribe_audio(command_audio)

print("Command:", command)

Listening...
Finished listening.
Command: i love you i love you so much


In [5]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

segments, info = model.transcribe(
    "debug_command.wav",
    language="en"
)

for segment in segments:
    print(segment.text)

 I love you, I love you so much


In [8]:
import sounddevice as sd

print(sd.query_devices())
print("Default device:", sd.default.device)

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
   1 Microphone (HD Pro Webcam C920), MME (2 in, 0 out)
   2 Microphone (Iriun Webcam #2), MME (2 in, 0 out)
*  3 Microphone (2- USB Audio Device, MME (1 in, 0 out)
   4 Microphone (Iriun Webcam), MME (2 in, 0 out)
   5 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
   6 Speakers (2- USB Audio Device), MME (0 in, 2 out)
   7 MSI G24C4 (NVIDIA High Definiti, MME (0 in, 2 out)
   8 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   9 Microphone (HD Pro Webcam C920), Windows DirectSound (2 in, 0 out)
  10 Microphone (Iriun Webcam #2), Windows DirectSound (2 in, 0 out)
  11 Microphone (2- USB Audio Device), Windows DirectSound (1 in, 0 out)
  12 Microphone (Iriun Webcam), Windows DirectSound (2 in, 0 out)
  13 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
  14 Speakers (2- USB Audio Device), Windows DirectSound (0 in, 2 out)
  15 MSI G24C4 (NVIDIA High Definition Audio), Windows DirectSound (0 in, 2 out)
 

In [3]:
import sounddevice as sd
import numpy as np

MIC_DEVICE = 3
SAMPLE_RATE = 16000

print("Speak for 5 seconds...")

audio = sd.rec(
    int(5 * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32",
    device=MIC_DEVICE
)

sd.wait()

audio = audio.flatten()

print("Max:", np.max(np.abs(audio)))
print("RMS:", np.sqrt(np.mean(audio ** 2)))
print("Min:", np.min(audio))

Speak for 5 seconds...
Max: 0.06411743
RMS: 0.010643986
Min: -0.06411743


In [9]:
from faster_whisper import WhisperModel
import sounddevice as sd
import re
from LLM import get_completion
from tts import speak
import numpy as np

SAMPLE_RATE = 16000
MIC_DEVICE = 19

# Initialize Whisper

model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)



# Record audio

def record_audio(duration):

    audio = sd.rec(
        int(duration * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        device=MIC_DEVICE
    )

    sd.wait()

    return audio.flatten()



# Transcribe

def transcribe_audio(audio):

    segments, info = model.transcribe(
        audio,
        language="en"
    )

    text = ""

    for segment in segments:
        text += segment.text

    text = re.sub(r"[^\w\s]", "", text)

    return text.lower().strip()




CHUNK_SIZE = 1024
SILENCE_THRESHOLD = 0.001
SILENCE_DURATION = 2


def listen_until_silence():

    print("Listening...")

    audio_chunks = []
    silence_time = 0

    while True:

        audio = sd.rec(
            CHUNK_SIZE,
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            device=MIC_DEVICE
        )

        sd.wait()

        audio = audio.flatten()

        rms = np.sqrt(np.mean(audio ** 2))

        print("RMS:", rms)

        audio_chunks.append(audio)

        if rms < SILENCE_THRESHOLD:
            silence_time += CHUNK_SIZE / SAMPLE_RATE
        else:
            silence_time = 0

        if silence_time >= SILENCE_DURATION:
            break

    audio = np.concatenate(audio_chunks)

    print("Recorded seconds:", len(audio) / SAMPLE_RATE)

    return audio








active = False

while True:

    if not active:

        print("Waiting for Hey Avanti...")

        audio = record_audio(5)

        text = transcribe_audio(audio)

        print("Heard:", text)

        if "hey avanti" in text:

            print("HEY AVANTI DETECTED!")

            active = True

            speak("How can I help you?")


    else:

        command_audio = listen_until_silence()

        command = transcribe_audio(command_audio)

        print("Command:", command)
        if not command:
            continue

        if "bye avanti" in command:

            print("BYE AVANTI")

            speak("Goodbye.")

            active = False

        else:

            response = get_completion(command)

            speak(response)



Waiting for Hey Avanti...


PortAudioError: Error opening InputStream: Invalid sample rate [PaErrorCode -9997]

In [6]:
import sounddevice as sd

print(sd.query_devices())
print("Default:", sd.default.device)

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Microphone (HD Pro Webcam C920), MME (2 in, 0 out)
   2 Microphone (Iriun Webcam #2), MME (2 in, 0 out)
   3 Microphone (2- USB Audio Device, MME (1 in, 0 out)
   4 Microphone (Iriun Webcam), MME (2 in, 0 out)
   5 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  6 Speakers (2- USB Audio Device), MME (0 in, 2 out)
   7 MSI G24C4 (NVIDIA High Definiti, MME (0 in, 2 out)
   8 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   9 Microphone (HD Pro Webcam C920), Windows DirectSound (2 in, 0 out)
  10 Microphone (Iriun Webcam #2), Windows DirectSound (2 in, 0 out)
  11 Microphone (2- USB Audio Device), Windows DirectSound (1 in, 0 out)
  12 Microphone (Iriun Webcam), Windows DirectSound (2 in, 0 out)
  13 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
  14 Speakers (2- USB Audio Device), Windows DirectSound (0 in, 2 out)
  15 MSI G24C4 (NVIDIA High Definition Audio), Windows DirectSound (0 in, 2 out)
 